In [1]:
# 使用A100进行训练
import os

os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

In [2]:
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForLanguageModeling, TrainingArguments, Trainer, BloomForCausalLM

/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
ds = Dataset.load_from_disk("./wiki_cn_filtered/")
ds

Dataset({
    features: ['source', 'completion'],
    num_rows: 10000
})

In [4]:
ds[0]

{'source': 'wikipedia.zh2307',
 'completion': "西安交通大学博物馆（Xi'an Jiaotong University Museum）是一座位于西安交通大学的博物馆，馆长是锺明善。\n历史\n2004年9月20日开始筹建，2013年4月8日正式建成开馆，位于西安交通大学兴庆校区陕西省西安市咸宁西路28号。建筑面积6,800平米，展厅面积4,500平米，馆藏文物4,900余件。包括历代艺术文物馆、碑石书法馆、西部农民画馆、邢良坤陶瓷艺术馆、陕西秦腔博物馆和书画展厅共五馆一厅。\n营业时间\n* 周一至周六：上午九点至十二点，下午一点至五点\n* 周日闭馆"}

In [5]:
tokenizer = AutoTokenizer.from_pretrained("Langboat/bloom-389m-zh")

def process_func(examples):
    contents = [e + tokenizer.eos_token for e in examples["completion"]]
    return tokenizer(contents, max_length=384, truncation=True)

In [6]:
tokenized_ds = ds.map(process_func, batched=True, remove_columns=ds.column_names)
tokenized_ds

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 10000
})

In [7]:
print(tokenized_ds[0])

{'input_ids': [13110, 34800, 13535, 916, 33156, 10, 256, 576, 387, 479, 681, 5453, 10955, 915, 24124, 5317, 13110, 6573, 20757, 13535, 355, 5358, 1490, 583, 28056, 1407, 3855, 671, 6113, 189, 6732, 4302, 9488, 3434, 6900, 1322, 355, 37336, 9825, 4608, 13461, 1359, 5358, 355, 5317, 13110, 34800, 4433, 7189, 25722, 29747, 13110, 1498, 12047, 6347, 23563, 2139, 2066, 420, 29288, 25, 15, 7635, 39288, 355, 1484, 5835, 6272, 23, 15, 4180, 39288, 355, 5358, 4516, 11621, 23, 15, 10641, 4887, 1712, 420, 2450, 31163, 8085, 11621, 5358, 553, 9888, 2731, 21335, 5358, 553, 9876, 14011, 4434, 5358, 553, 21484, 4514, 17170, 25871, 8085, 5358, 553, 17489, 6945, 11097, 13535, 641, 33623, 1484, 5835, 1689, 2063, 5358, 569, 5835, 671, 16225, 3422, 189, 13, 23158, 27894, 33227, 1022, 11396, 3347, 1813, 1504, 6566, 1813, 355, 9155, 8633, 1504, 2063, 1813, 189, 13, 23158, 813, 7817, 5358, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1

In [8]:
from torch.utils.data import DataLoader

dl = DataLoader(tokenized_ds, batch_size=16, collate_fn=DataCollatorForLanguageModeling(tokenizer, mlm=False))

In [9]:
next(enumerate(dl))

(0,
 {'input_ids': tensor([[    3,     3,     3,  ...,  7817,  5358,     2],
         [  586, 11835,   739,  ..., 10981, 21350,  9067],
         [    3,     3,     3,  ...,  4412,   420,     2],
         ...,
         [    3,     3,     3,  ..., 22357,   420,     2],
         [13904,  3867,  8840,  ..., 11142,  3416,   355],
         [  816,  3193,  1038,  ..., 22676,  6140, 13165]]), 'attention_mask': tensor([[0, 0, 0,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1],
         [0, 0, 0,  ..., 1, 1, 1],
         ...,
         [0, 0, 0,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1]]), 'labels': tensor([[ -100,  -100,  -100,  ...,  7817,  5358,     2],
         [  586, 11835,   739,  ..., 10981, 21350,  9067],
         [ -100,  -100,  -100,  ...,  4412,   420,     2],
         ...,
         [ -100,  -100,  -100,  ..., 22357,   420,     2],
         [13904,  3867,  8840,  ..., 11142,  3416,   355],
         [  816,  3193,  1038,  ..., 22676,  6140, 

In [10]:
tokenizer.pad_token, tokenizer.pad_token_id

('<pad>', 3)

In [11]:
tokenizer.eos_token, tokenizer.eos_token_id

('</s>', 2)

In [12]:
model = AutoModelForCausalLM.from_pretrained("Langboat/bloom-389m-zh")

In [13]:
args = TrainingArguments(
    output_dir="./causal_lm",
    per_device_train_batch_size=16,
    gradient_accumulation_steps=4,
    logging_steps=20,
    num_train_epochs=5
)

In [14]:
trainer = Trainer(
    args=args,
    model=model,
    tokenizer=tokenizer,
    train_dataset=tokenized_ds,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

/tmp/ipykernel_2513/1098143094.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Detected kernel version 4.15.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


[2025-12-03 10:11:10,537] [INFO] [real_accelerator.py:254:get_accelerator] Setting ds_accelerator to cuda (auto detect)
[2025-12-03 10:11:12,205] [INFO] [logging.py:107:log_dist] [Rank -1] [TorchCheckpointEngine] Initialized with serialization = False


In [15]:
trainer.train()

Step,Training Loss
20,4.184900
40,3.987000
60,7.066400
80,6.515800
100,3.896100
120,3.675300
140,3.576500
160,3.520800
180,3.336800
200,3.867400


TrainOutput(global_step=785, training_loss=4.010721330581957, metrics={'train_runtime': 3380.2329, 'train_samples_per_second': 14.792, 'train_steps_per_second': 0.232, 'total_flos': 3.48265119744e+16, 'train_loss': 4.010721330581957, 'epoch': 5.0})

In [16]:
from transformers import pipeline

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, device=0)

Device set to use cuda:0


In [17]:
pipe("西安交通大学博物馆（Xi'an Jiaotong University Museum）是一座位于西安", max_length=128, do_sample=True)

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Both `max_new_tokens` (=256) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': "西安交通大学博物馆（Xi'an Jiaotong University Museum）是一座位于西安市西郊的藏传性博物馆，于2018年12月30日启用，由中华人民共和国国务院文物局管理，是陕西省唯一一家藏传性博物馆，位于西安市西安市西郊，占地面积约5万平方米。该博物馆是以藏传文化、文化遗产和考古、文物、艺术、历史文化、地方史、考古文化、考古研究为特色，是陕西省、西安市文物保护单位、陕西省历史文化研究的重要组成部分。\n历史\n西汉初\n*西汉西汉初，西汉皇帝刘邦被刘邦录取为继承人，刘邦和刘邦夫人为西汉第一位皇后。\n刘邦之子刘邦有才智谋远见刘邦，刘邦非常敬畏刘邦，刘邦非常担心刘邦会死，刘邦非常害怕刘邦会杀死刘邦，刘邦认为刘邦会死。刘邦到西汉首都刘邦城，刘邦在刘邦城内找到刘邦的坟墓，刘邦与刘邦夫人刘邦、刘邦曾孙刘邦的墓葬在汉朝城。刘邦将墓葬移到汉朝城，并在刘邦墓旁立有汉朝城最早的碑和汉朝城最早的图，现在刘邦墓旁立有一尊汉朝城最早的壁画，壁画上描绘了刘邦墓前场景。刘邦因刘邦墓葬在汉朝城，故汉朝城得名，汉朝城得名"}]

In [18]:
pipe("下面是一则游戏新闻。小编报道，近日，游戏产业发展的非常", max_length=128, do_sample=True)

Both `max_new_tokens` (=256) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': '下面是一则游戏新闻。小编报道，近日，游戏产业发展的非常快，《王者荣耀》这个游戏一鸣大动。对于玩家来说，这个游戏是王者荣耀里面最受欢迎的游戏之一，而且他的受欢迎程度是游戏行业内最受欢迎的。\n王者荣耀\n王者荣耀是腾讯和网易合作推出的手机游戏，由腾讯和网易公司联合出品。腾讯和网易于2019年5月共同发布，并于6月27日在腾讯视频网站上上线。王者荣耀的评价褒贬不一，有赞有贬。\n王者荣耀作为腾讯和网易合作推出的一款游戏，它受到了广大玩家们的喜爱，游戏玩法简单，玩法多样，具有很长的社交圈，社交圈内的场景也精彩，社交圈内往往会设大量道具道具，可以和玩家展开各种各样的社交圈。\n游戏玩法\n王者荣耀一共有8种不同的职业，每种都有独特的操作方式和技能。玩家可以根据情况来选择哪种职业，再根据情况来选择一种技能，比如：\n* 英雄\n* 战士\n* 刺客\n* 刺客\n* 刺客\n* 刺客\n* 刺客\n* 刺客\n* 刺客\n* 刺客\n* 刺客\n* 刺客\n* 刺客\n* 刺客\n* 刺客\n* 刺客\n* 刺客\n* 刺客\n'}]